## 9. Coseismic slip inversion

`nisar_tools.slip` takes the `LOSStack` from §6 and inverts it for slip on a fault, discretised into triangular dislocation elements and solved as a bounded, smoothed linear least-squares problem. It is a Python port of [SlipSolve-Curve](https://github.com/x3zou/SlipSolve-Curve).

The flow is:

```
FaultTrace  →  LocalFrame  →  FaultMesh.vertical  →  Observations.from_los  →  SlipInversion  →  SlipModel
 (.kml)        (shared!)       (the fault)            (quadtree sampling)       (Green's + solve)   (.save)
```


Three modelling choices are independent, and the simplest of each is the default. §9.1–9.3 below show the alternatives; everything between here and there uses the defaults.

| | default | also available |
|---|---|---|
| geometry | vertical (`FaultMesh.vertical`) | curved, one dip per deep segment (`FaultMesh.curved`) |
| medium | homogeneous half-space (`HalfSpaceTDE`) | layered, from EDGRN tables (`LayeredPointSource`) |
| slip basis | constant per element | continuous nodal tent functions (`basis="node"`) |

Four things that are easy to get wrong:

- **One `LocalFrame` for everything.** A transverse Mercator centred on the study area, *not* UTM, so two tracks in different zones share one x/y. Every object stores its frame and checks it — mixing frames is a silent kilometre-scale error.
- **Positive `strike_slip` is LEFT-lateral.** A right-lateral fault (San Sebastián, Sagaing) needs `polarity=(-1, 0, 0)`.
- **`exclude_within` is required.** Dislocation solutions are singular *on* the fault surface, so a sample sitting on the trace gives a non-finite Green's function and the inversion refuses to run rather than quietly zeroing it.
- **One track per scene, plus `ramp=`.** Every unwrapped interferogram carries an arbitrary constant and usually an orbital/ionospheric ramp. `ramp="linear"` gives each *named* track its own offset and `x`/`y` gradients; without it, those land in the slip as broad, deep, entirely fictitious patches.

In [ ]:
import os
from pathlib import Path

from nisar_tools.slip import (
    FaultTrace, FaultMesh, Observations, SlipInversion, SlipModel,
)

# A fault trace: .kml (Google Earth) or two-column lon/lat ASCII.
FAULT = Path(os.environ.get("NISAR_FAULT", "~/Downloads/fault_trace.kml")).expanduser()

if not FAULT.exists():
    trace = None
    print(f"No fault trace at {FAULT}\n-> skipping section 9 (set NISAR_FAULT to run it).")
else:
    trace = FaultTrace.from_file(FAULT)
    # ONE frame, shared by the mesh and by every track's observations.
    frame = trace.local_frame()
    mesh = FaultMesh.vertical(trace, frame, max_depth=20e3, edge_length=3e3)
    print(trace)
    print(mesh, f"-> {2 * mesh.n_elements} slip parameters")

### Measure the sampling parameters, per scene

The three numbers `Observations.from_los` takes are statements about the **data**, not preferences — and inheriting them from an example is how a setup goes quietly wrong:

- **`rms_min` is a noise level.** The quadtree splits while the *pixel* scatter inside a cell exceeds it, and that scatter does not shrink as the cell does. Set below the noise, cells can never stop splitting; they just run down to `width_min`, so the sample count is set by a size limit rather than by information and most of what you sample is atmosphere.
- **`width_min` is not continuous.** Cells are index rectangles halved at their midpoint, so the sizes actually reachable form a dyadic ladder — per axis, and different for every scene. Changing it does nothing at all until it crosses a rung, and then changes the sample count by a factor of two.
- **`exclude_within` only has to stop a *cell* straddling the trace**, where the displacement field is discontinuous and the dislocation solution is undefined. That floor is `width_min / 2`; larger values are a judgement about unwrapping errors and near-fault model error, not a requirement.

`scene_report` measures all three from the scene itself. It also reports the thing that actually limits the answer, which no parameter can fix: **coverage on each side of the fault, along strike**. An aggregate percentage hides the failure that matters — a fault can look adequately covered on average while one block is missing over precisely the stretch where the rupture is largest.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from nisar_tools import LOSStack, Workspace
from nisar_tools.slip import ramp_content, scene_report
from nisar_tools.slip.plot import (
    plot_coverage, plot_fit, plot_l_curve, plot_samples, plot_slip,
)

WORK_DIR = Path(os.environ.get("NISAR_WORK_DIR", "workdir")).expanduser()
ws = Workspace(WORK_DIR)
los = LOSStack.from_zarr(ws.path("los"))
if trace is not None:
    # One entry per scene. Add the ascending track the same way -- a second
    # look direction is what separates vertical from east-west motion.
    scenes = {"D071": los}          # e.g. {"D071": los, "A014": los_asc}

    reports = {name: scene_report(stack, trace, frame, mesh=mesh)
               for name, stack in scenes.items()}

    display(pd.DataFrame({name: {
        "noise floor (mm)":     round(1e3 * r.attrs["noise_floor"], 1),
        "-> rms_min (m)":       round(r.attrs["rms_min"], 4),
        "-> width_min (m)":     round(r.attrs["width_min"]),
        "-> exclude_within (m)": round(r.attrs["exclude_within"]),
        "smallest cell (m)":    round(r.attrs["terminal_cell"]),
        "left of trace (%)":    round(100 * r.attrs["frac_left"], 1),
        "two-sided (%)":        round(100 * r.attrs["two_sided_fraction"], 1),
        "near-fault cover (%)": round(100 * r.attrs["near_fault_coverage"], 1),
        "geometry OK":          r.attrs["geometry_consistent"],
    } for name, r in reports.items()}).T)

### Sample the LOS rasters — one track per scene

A full-resolution LOS raster has millions of pixels against a few thousand slip parameters, so it is quadtree-downsampled first: cells split where the signal varies and stay coarse where it doesn't.

The parameters come straight from the reports above rather than from literals. Each scene gets its **own `name`**, and `Observations.concat` combines them — that name is what `ramp=` keys the nuisance columns on, so keeping scenes separate here is what lets each carry its own arbitrary offset and ramp.

In [ ]:
if trace is not None:
    obs = Observations.concat(
        [
            Observations.from_los(
                stack, name=name, frame=frame, trace=trace,
                rms_min=reports[name].attrs["rms_min"],
                width_min=reports[name].attrs["width_min"],
                exclude_within=reports[name].attrs["exclude_within"],
                width_max=30_000.0,       # coarsest cell; rarely the binding limit
            )
            for name, stack in scenes.items()
        ],
        normalize="sqrt_count",           # equal total influence per track
    )
    print(obs, f"-> {obs.n / (2 * mesh.n_elements):.1f}x the slip parameters")
    fig, ax = plot_samples(obs, trace=trace)   # marker size = quadtree cell size

    # Is `ramp=` absorbing nuisance, or signal? For a long, near east-west
    # strike-slip fault the far-field coseismic pattern is an arctangent step
    # across the trace, which over a finite aperture looks a lot like a gradient
    # perpendicular to strike -- so the ramp columns and the slip genuinely
    # compete. The columns are normalised by each track's span, so a gradient
    # reads directly as metres of LOS across the scene: centimetres is orbit,
    # tens of centimetres is deformation being taken away from the slip model.
    ramp = ramp_content(obs)
    print(f"\nexplained by a per-track offset alone : "
          f"{ramp['offset']['variance_reduction']:5.1f}%")
    print(f"          ...plus x/y gradients       : "
          f"{ramp['linear']['variance_reduction']:5.1f}%"
          f"   (+{ramp['gradient_only']:.1f} from the gradients)")
    print("gradients (m of LOS across the scene):",
          {k: round(v, 3) for k, v in ramp["linear"]["coefficients"].items()
           if k.endswith((":dx", ":dy"))})

### Solve

`SlipInversion` builds the Green's matrix once — column *k* is what every observation would record for one metre of slip on element *k* — and `solve` runs a bounded, smoothed least-squares fit. The Green's matrix is the expensive part and is reused across every smoothing weight, which is what makes the L-curve sweep below cheap.

In [ ]:
if trace is not None:
    # ramp="linear": each track gets its own offset + x/y gradients, so the
    # interferogram's arbitrary constant and orbital ramp don't become slip.
    inv = SlipInversion(mesh, obs, ramp="linear")
    print(inv)

    model = inv.solve(
        smoothing=0.3,
        polarity=(-1, 0, 0),      # right-lateral: strike-slip pinned non-positive
        strike=(-6.0, 6.0),       # bounds, metres
        dip=(-1.0, 1.0),
    )
    print(model)
    print("converged:", model.converged, "| iterations:", model.result.nit)
    print("ramp terms:", dict(zip(model.ramp_labels, model.ramp.round(4))))

    fig, ax = plot_slip(model)                  # unrolled fault: along-strike vs depth
    fig, axes = plot_fit(model, trace=trace)    # data / model / residual per track

### Sub-sampling from the model, on one common lattice

The model above is the "approximate model" to start from. Everything so far chose
quadtree cells from the **observed** displacement. That is
noise-driven: the split test is the *pixel* scatter inside a cell, which does not
shrink as the cell shrinks, so once `rms_min` sits below the noise the recursion
cannot stop on information and simply runs down to `width_min`.

**First, one lattice.** A quadtree cell is an integer number of pixels halved at its
midpoint, so the reachable cell sizes are a dyadic ladder *set by the pixel size* —
per axis, per scene. Two scenes at different resolutions land on different ladders,
one `width_min` means two different things, and their sample counts diverge for a
reason that has nothing to do with the data. `resample_all` puts every track on one
grid in the shared `LocalFrame`, defaulting to **10 arc-seconds** (309 m) — the usual
ALOS-2 posting, and already ~20× finer than a fault element.

**Then the loop.** Round 0 is deliberately coarse and data-driven; each later round
predicts each scene's LOS, re-samples on that, and re-solves, stopping when no
parameter moves by more than `tol`. `refine_within` holds a dense band along the trace
through every round — without it an initial model with little shallow slip predicts a
smooth near field, the quadtree coarsens precisely where shallow slip needs
constraining, and the next round is free to invent it.

In [ ]:
from nisar_tools.slip import (
    ARCSEC_10, iterate_sampling, model_rms_min, predicted_los, resample_all,
)
from nisar_tools.slip.plot import plot_mesh

if trace is not None:
    # One lattice for every track: 10 arcsec, in the frame the inversion works in.
    gridded = resample_all(scenes, frame, spacing=ARCSEC_10)
    for name, stack in gridded.items():
        print(f"{name}: {dict(stack.ds.sizes)} @ {ARCSEC_10:.1f} m in the local frame")

    sampling = {
        name: dict(rms_min=reports[name].attrs["rms_min"],
                   width_min=reports[name].attrs["width_min"],
                   width_max=30_000.0,
                   exclude_within=reports[name].attrs["exclude_within"])
        for name in gridded
    }

    # Round 0 is coarse and data-driven; every later round is driven by the model.
    obs, model, history = iterate_sampling(
        gridded, mesh, trace, frame, sampling,
        max_rounds=4, spacing=2000.0,
        inversion_kwargs={"ramp": "linear"},
        solve_kwargs={"smoothing": 0.3, "polarity": (-1, 0, 0),
                      "strike": (-6.0, 6.0), "dip": (-1.0, 1.0)},
    )
    display(pd.DataFrame(history).set_index("round"))

    # `obs` and `model` now supersede the data-driven pair from above. Take the
    # Green's matrix with them -- it is the expensive part and it is already built,
    # so the L-curve below sweeps the *refined* sampling for free.
    inv = model.inversion

    # The two figures to look at before committing to a layered run: does the mesh
    # resolve what the samples can constrain, and do the cells shrink toward the fault?
    plot_mesh(mesh, trace=trace, color="area")
    plot_samples(obs, trace=trace)
    plot_fit(model, trace=trace)   # residual is observed minus modelled

### Choosing the smoothing weight: the L-curve

Too much smoothing and the model can't fit the data; too little and it fits the noise with rough, deep, physically implausible slip. Plotting misfit against roughness across many weights gives a curve with a corner — that corner is the conventional choice.

The Green's matrix is built once and reused across every weight, which is what makes the sweep affordable.

In [ ]:
if trace is not None:
    # Swept large -> small internally: a smoother problem is better conditioned,
    # and the sweep's cost is dominated by its roughest end.
    curve, models = inv.l_curve(
        [2.0, 1.0, 0.5, 0.3, 0.1, 0.05, 0.02, 0.01],
        polarity=(-1, 0, 0), strike=(-6.0, 6.0), dip=(-1.0, 1.0),
    )
    display(curve[["rms_misfit", "roughness", "variance_reduction",
                   "iterations", "converged"]].to_dataframe())

    fig, ax = plot_l_curve(curve)

    # Weights that hit the cap have meaningless statistics AND dominate the
    # sweep's runtime -- drop them rather than trying to speed them up.
    capped = curve["smoothing"].values[~curve["converged"].values]
    if capped.size:
        print(f"WARNING: {list(capped)} hit the iteration cap -- their "
              "statistics are meaningless. Drop them, raise max_iter, or "
              "stay above the roughest weight that converges.")

### Saving the result

`model.save(path)` writes **one self-contained file** — the slip vector, the fit, the mesh and the observations — that can be copied off the machine that produced it. `SlipModel.load` gives back a model that reports every statistic, re-exports, plots, and forward-models new points.

The Green's matrix is deliberately *not* saved: it is the largest object in the problem and nothing downstream of a solved model needs it. A loaded model therefore cannot be re-solved at a new weight — rebuild the `SlipInversion` for that.

`to_text` writes SlipSolve's ten-column element table (`element_id lon lat depth strike dip strike_slip dip_slip area mu`), so existing GMT scripts work unchanged.

In [ ]:
if trace is not None:
    model.save(WORK_DIR / "slip_model.slip.zip")
    model.to_text(WORK_DIR / "slip_model.txt")     # GMT-ready element table

    # ...later, or in a fresh kernel, with no mesh or observations to rebuild:
    reloaded = SlipModel.load(WORK_DIR / "slip_model.slip.zip")
    print(reloaded)
    print("mesh matches:", reloaded.mesh.digest() == mesh.digest())
    print("tracks      :", reloaded.obs.tracks)
    fig, ax = plot_slip(reloaded)

### Running a long inversion in the background

A fine mesh against a dense quadtree runs for a long time, and a notebook kernel is a poor place to leave it — closing the browser or losing the SSH session takes the run with it.

`scripts/` holds three ready-to-run stages for exactly this — `run_sampling.py`, `run_lcurve.py` and `run_inversion.py`, all configured from `scripts/slip_config.py`. See **"Running an inversion in the background"** in the [README](../README.md) for the launch recipe, the environment-variable overrides (mesh size, dip, smoothing weight) and the three details that make a detached job work: `-u`, `KMP_DUPLICATE_LIB_OK=TRUE`, and the env's absolute interpreter path.

Pick the result up back here afterwards:

```python
model = SlipModel.load(WORK_DIR / "model_sampling" / "slip_model.slip.zip")
```

### 9.1 A fault that dips

`FaultMesh.vertical` extrudes the trace straight down. A real fault usually doesn't go straight down, and a vertical mesh has nowhere to put dip-slip signal except into strike-slip or the residual.

`FaultMesh.curved` follows the reference implementation's workflow: give it a set of straight **deep segments** in plan view and **one dip each**. Each segment is pushed down-dip by `depth / tan(dip)` along the trace's normal, a smooth surface is fitted through those bottom lines *and* the surface trace, and the mesh nodes are that surface sampled on the (along-strike, depth) lattice.

A segment file holds four numbers — `x_begin y_begin x_end y_end`, in **metres in the local frame**. `FaultSegment.from_trace(trace, frame, n)` chops the trace into `n` equal chords instead, which is enough when all you want to say is "the western third dips 70°, the rest 85°".

Things worth knowing:

- **The surface between the trace and the bottom lines is decided by the regularizer**, because nothing else constrains it — only two depths carry control points. That is why `smoothness` matters and why the reference's specific gridder is ported rather than substituted. Pass `depth_control=(x, y, depth)` points (relocated seismicity, say) if you want the profile to bend with depth.
- **Dips above 90° are meaningful, not an error**: the fault leans the other way. The reference's Myanmar configuration uses `[75 75 70 80 85 90 100]`.
- **`bias_w` thickens the depth levels downward**, putting fine resolution where surface data can actually constrain slip. A patch at 2 km is resolved far more sharply than one at 18 km, so uniform levels spend parameters where they cannot be recovered.

In [ ]:
import numpy as np

from nisar_tools.slip import FaultSegment

if trace is not None:
    # Three deep segments straight off the trace, dipping progressively steeper.
    # In a real setup these come from files: FaultSegment.from_files([...]).
    segments = FaultSegment.from_trace(trace, frame, 3)
    for seg in segments:
        print(seg)

    mesh_dip = FaultMesh.curved(
        trace, frame,
        segments=segments,
        dips=[70.0, 80.0, 85.0],   # one per segment; >90 leans the other way
        max_depth=20e3,
        edge_length=3e3,
        bias_w=1.15,               # depth levels thicken downward
        smoothness=0.008,          # the reference's surfaceFitSmoothness
    )
    print("\n", mesh_dip)
    print(f"  dip         {mesh_dip.dip.min():.1f} - {mesh_dip.dip.max():.1f} deg")
    print(f"  dip azimuth {np.median(mesh_dip.dip_direction):.0f} deg "
          "(`dip` folds to [0,90], so read this to tell the two leans apart)")

    # How far the base has stepped off the trace, per along-strike position.
    n_along = mesh_dip.attrs["n_along"]
    step = np.hypot(*(mesh_dip.nodes[-n_along:, :2] - mesh_dip.nodes[:n_along, :2]).T)
    print(f"  base offset {step.min() / 1e3:.1f} - {step.max() / 1e3:.1f} km "
          "(shallow dip steps further)")

    # A single dip everywhere, and the vertical special case:
    #   FaultMesh.curved(trace, frame, uniform_dip=75.0, max_depth=20e3)
    #   FaultMesh.curved(trace, frame, uniform_dip=90.0)  == FaultMesh.vertical(...)

### 9.2 A layered crust, and nodal slip

**A layered medium.** A homogeneous half-space gives the whole crust one rigidity, and the shallowest few kilometres are much softer than that. Assuming otherwise means the same surface displacement needs *less* shallow slip and *more* deep slip — a systematic bias in exactly the quantity a coseismic inversion is for. `LayeredPointSource` cuts each element into point sources and looks each one up in Green's-function tables from Rongjiang Wang's **EDGRN**:

```python
tables = EdgrnTables.from_input_file("edgrn.inp")   # tables you generated
engine = LayeredPointSource(tables)
model  = SlipInversion(mesh, obs, engine=engine).solve(smoothing=0.3)
```

**Nodal slip.** `basis="node"` solves for slip at the mesh *nodes*, with a continuous piecewise-linear field between them, instead of a constant value per triangle. A real slip distribution is continuous, and a piecewise-constant one spends resolution representing edges that are not there. There are fewer nodes than triangles, so it is also a *smaller* problem, and it needs a smoothing operator defined on the surface (a Laplace–Beltrami operator) rather than on element adjacency — `solve` picks the right one automatically.

Slip is still **reported per element** (`model.element_slip`, `to_dataset`, `to_text`, `plot_slip`), so nothing downstream changes with the parameterization.

**Depth-dependent rigidity.** Once you have a `VelocityModel`, `model.moment(crust)` samples `mu = rho·vs²` at each parameter's own depth instead of assuming a single 30 GPa.

In [ ]:
from nisar_tools.slip import EdgrnTables, LayeredPointSource, VelocityModel

if trace is not None:
    # A layered crust: soft at the top, stiffening with depth. In a real run this
    # is `VelocityModel.from_file("crust.txt")` -- depth vp vs rho, in SI.
    crust = VelocityModel(
        depth=[0.0, 5e3, 15e3, 30e3],
        vp=[4.0e3, 5.5e3, 6.3e3, 6.8e3],
        vs=[2.3e3, 3.2e3, 3.6e3, 3.9e3],
        rho=[2.4e3, 2.6e3, 2.8e3, 3.0e3],
    )
    print(crust, f"| nu at the surface {crust.poisson():.3f}")

    # Real tables come from EDGRN:
    #   tables = EdgrnTables.from_input_file(WORK_DIR / "edgrn.inp")   # you ran it
    #   tables = run_edgrn(crust, WORK_DIR / "edgrn")                  # we run it
    # Here: synthesised tables for a UNIFORM medium, whose answer must reproduce
    # the homogeneous half-space engine -- the check worth having when there is no
    # Fortran to hand. Takes a few seconds.
    tables = EdgrnTables.homogeneous(
        nu=0.25,
        r=np.linspace(0.0, 300e3, 301),      # epicentral distance, 1 km cells
        z=np.linspace(250.0, 30e3, 120),     # source depth
    )
    print(tables)

    engine = LayeredPointSource(tables, tolerance=3e-3)
    inv_layered = SlipInversion(mesh, obs, engine=engine, ramp="linear", basis="node")
    print(f"\n{inv_layered}")
    print(f"  parameters: {inv_layered.n_param} (nodal) vs "
          f"{2 * mesh.n_elements + inv_layered.n_ramp} (element-constant)")

    model_layered = inv_layered.solve(
        smoothing=0.3, polarity=(-1, 0, 0), strike=(-6.0, 6.0), dip=(-1.0, 1.0),
    )
    print(f"  {model_layered}")
    print(f"  Mw with a layered crust : "
          f"{2 / 3 * (np.log10(model_layered.moment(crust)) - 9.1):.2f}")
    print(f"  Mw assuming 30 GPa      : "
          f"{2 / 3 * (np.log10(model_layered.moment(30e9)) - 9.1):.2f}")

    # Slip is reported per element whatever the basis, so plotting is unchanged.
    fig, ax = plot_slip(model_layered)

### 9.3 Exporting the model's surface displacement (Ux, Uy, Uz)

A radar measures one number per pixel: the projection of a three-component ground motion onto the line of sight. What the inversion recovers is the **full vector**, and `surface_displacement` evaluates it on a regular grid — `ux` east, `uy` north, `uz` up, in metres, positive in those directions.

`to_grd` writes one GMT `.grd` per component, reprojected to lon/lat by the same `write_grd` every other stage exports through — so these drop into an existing GMT or `pygmt` workflow beside the `.grd` files from §7.

Two practical points:

- **Grid points on the trace come back NaN.** A dislocation solution is genuinely singular where the fault meets the free surface, and returning a very large number there would set the colour scale of every plot of the field. `exclude_within` defaults to the mesh's own element size — the scale below which a discretised fault does not mean anything anyway.
- **It evaluates in blocks.** The engines build a `(points, 3, 2·n_elements)` array, and a 1 km grid over a real footprint would ask for gigabytes of it — 60 000 points against 1148 elements is 3.3 GB. Raise `spacing` for a quick look; the default 1 km is already fine.

In [ ]:
if trace is not None:
    # `model` is the phase-1 result from the "Solve" cell above -- vertical fault,
    # homogeneous half-space -- so this works on the simplest possible model.
    field = model.surface_displacement(spacing=2000.0, pad=60e3)
    print(field)

    peak = {name: float(np.nanmax(np.abs(field[name]))) for name in ("ux", "uy", "uz")}
    print("\npeak |displacement| (m):", {k: round(v, 4) for k, v in peak.items()})
    print(f"horizontal / vertical  : "
          f"{max(peak['ux'], peak['uy']) / peak['uz']:.0f}x  "
          "(strike-slip barely moves the ground up or down)")

    fig, axes = plt.subplots(1, 3, figsize=(16, 4), constrained_layout=True)
    limit = max(peak.values())
    for ax, name, label in zip(axes, ("ux", "uy", "uz"),
                               ("east (Ux)", "north (Uy)", "up (Uz)")):
        field[name].plot(
            ax=ax, cmap="RdBu_r", vmin=-limit, vmax=limit,
            cbar_kwargs={"label": "displacement (m)"},
        )
        ax.plot(*trace.to_local(frame), "k-", lw=1.2)   # the fault trace
        ax.set_title(label)
        ax.set_aspect("equal")
        ax.set_xlabel("Local x (m)")
        ax.set_ylabel("Local y (m)")

    # One GMT `.grd` per component, reprojected to lon/lat.
    for path in model.to_grd(WORK_DIR / "grd_slip", spacing=2000.0, pad=60e3):
        print(path)
    print(f"\n  gmt grdimage {WORK_DIR / 'grd_slip' / 'ux.grd'} "
          "-JM6i -Baf -Cpolar -png ux")